# Preparación de datos

**Conjunto de datos:** Dataset 8 - Matriz Estación - Empresa

**Nombre de archivo:** stations_permits_matrix.csv

## 0. Inicialización

Instalar ydata-profiling

In [67]:
!pip install ydata-profiling

Instalar haversine

In [68]:
!pip install haversine

Importaciones

In [69]:
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import seaborn as sns

from haversine import haversine, Unit
from tabulate import tabulate

Visualización de tablas y gráficas

In [70]:
sns.set_style("darkgrid")

def print_table(df):
    print(tabulate(df, headers='keys', tablefmt='simple_outline'))

Lectura y muestra del archivo

In [71]:
stations = pd.read_csv('../data/original/air_quality_stations.csv')
permits = pd.read_csv('../data/preparada/emission_permits.csv')
measurements = pd.read_csv('../data/preparada/measurements.csv')

**1.** Eliminar columnas

In [72]:
stations = stations[['id', 'latitude', 'longitude']]
stations

,id,latitude,longitude
0,8,4.530214,-74.142217
1,18,4.940316,-73.970698
2,20,5.200000,-73.883000
3,4,4.596229,-74.194715
4,6,4.695702,-74.215583
5,17,4.862871,-74.056341
6,16,4.298055,-74.819167
7,23,4.594991,-74.204826
8,14,4.716525,-74.211712
9,22,4.809444,-74.102500


In [73]:
permits = permits[['ID', 'Latitud', 'Longitud']]
permits

,ID,Latitud,Longitud
0,1,4.703418,-74.226561
1,2,5.318407,-73.704281
2,3,4.800462,-74.210355
3,4,4.678797,-74.284112
4,5,4.699590,-74.193752
...,...,...,...
531,540,5.496933,-73.625257
532,541,4.274013,-74.446186
533,542,5.228254,-73.817856
534,543,4.303990,-74.803157


In [74]:
measurements = measurements[['station', 'variable']]
measurements

,station,variable
0,3,WDS
1,2,NO
2,18,RAIN
3,18,WDS
4,3,Temp
...,...,...
380091,14,WDS
380092,14,PM10
380093,9,PM10
380094,4,SO2


**2.** Eliminar duplicados

In [75]:
measurements = measurements.drop_duplicates()
measurements

,station,variable
0,3,WDS
1,2,NO
2,18,RAIN
3,18,WDS
4,3,Temp
...,...,...
188612,8,SO2
192862,10,NO
192880,10,PM10
192884,10,NOX


**3.** Renombrar columnas

In [76]:
stations = stations.rename(columns={
    'id': 'IDEstación',
    'latitude': 'LatitudEstación',
    'longitude': 'LongitudEstación'
})

permits = permits.rename(columns={
    'ID': 'IDEmpresa',
    'Latitud': 'LatitudEmpresa',
    'Longitud': 'LongitudEmpresa'
})

measurements = measurements.rename(columns={
    'station': 'IDEstación',
    'variable': 'Variable'
})

**4.** Guardar distancias de estaciones a empresas cuando:

- Tiene una distancia menor o igual a 20 kms a la redonda

In [77]:
# Crear el producto cartesiano de todas las estaciones con todos los permisos
arcos = stations[['IDEstación', 'LatitudEstación', 'LongitudEstación']].merge(
    permits[['IDEmpresa', 'LatitudEmpresa', 'LongitudEmpresa']], 
    how='cross'
)

# Calcular la distancia haversine para cada par
arcos['DistanciaKm'] = arcos.apply(
    lambda row: haversine(
        (row['LatitudEstación'], row['LongitudEstación']),
        (row['LatitudEmpresa'], row['LongitudEmpresa']),
        unit=Unit.KILOMETERS
    ),
    axis=1
)

# Filtrar solo los arcos con distancia <= 20 km
df = arcos[arcos['DistanciaKm'] <= 20].copy()

# Mantener solo las columnas de IDs y distancia
df = df[['IDEstación', 'IDEmpresa', 'DistanciaKm']]

# Resetear el índice
df = df.reset_index(drop=True)
df

,IDEstación,IDEmpresa,DistanciaKm
0,8,5,19.680815
1,8,20,9.812424
2,8,36,9.701665
3,8,39,9.954586
4,8,41,9.898695
...,...,...,...
2375,5,528,10.709926
2376,5,529,10.719429
2377,5,530,10.743336
2378,5,531,10.690984


**5.** Obtener variables medidas por la estación, creando un arco por cada variable

In [78]:
df = df.merge(
    measurements, 
    how='inner',
    on='IDEstación'
)
df

,IDEstación,IDEmpresa,DistanciaKm,Variable
0,8,5,19.680815,NO2
1,8,5,19.680815,NO
2,8,5,19.680815,PM2.5
3,8,5,19.680815,SO2
4,8,20,9.812424,NO2
...,...,...,...,...
13107,5,544,10.770024,SO2
13108,5,544,10.770024,CO
13109,5,544,10.770024,WDD
13110,5,544,10.770024,RAIN


**13.** Ordenar por IDs

In [79]:
df.sort_values(by=['IDEstación', 'IDEmpresa', 'Variable'], inplace=True)

Resultado final

In [80]:
df

,IDEstación,IDEmpresa,DistanciaKm,Variable
10346,2,1,5.502287,NO
10354,2,1,5.502287,NO2
10353,2,1,5.502287,NOX
10348,2,1,5.502287,PM10
10347,2,1,5.502287,PM2.5
...,...,...,...,...
4354,23,544,11.505391,PM2.5
4356,23,544,11.505391,RAIN
4355,23,544,11.505391,Temp
4353,23,544,11.505391,WDD


In [81]:
reporte = ProfileReport(df)
reporte.to_file("archivos_generados/Reporte perfilamiento - Dataset 8.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 199.89it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Exportar a CSV

In [82]:
df.to_csv('../data/preparada/stations_permits_matrix.csv', index=False)